# Experiment 1


In [ ]:
import os, shutil, time
# Create an "original" piece of evidence
with open("evidence_original.txt", "w") as f:
f.write("Case File #001 - Suspect device log")
orig_stat = os.stat("evidence_original.txt")
print("Original file - created (ctime):", time.ctime(orig_stat.st_ctime))
print("Original file - modified (mtime):", time.ctime(orig_stat.st_mtime))
# Simulate an investigator copying the file to a workstation
time.sleep(1)
shutil.copy2("evidence_original.txt", "evidence_copy.txt")
copy_stat = os.stat("evidence_copy.txt")
print("\nCopied file - created (ctime):", time.ctime(copy_stat.st_ctime))
print("Copied file - modified (mtime):", time.ctime(copy_stat.st_mtime))
print("\nLocard's Exchange Principle: the copy operation itself created a new")
print("ctime on the destination file - every interaction leaves a trace.")

# Experiment 2


In [ ]:
import hashlib, datetime
def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

# Step 1: Investigator seizes evidence and hashes it immediately
with open("evidence.txt", "w") as f:
    f.write("Suspect chat log: meeting at 10pm, bring the drive.")
seizure_hash = sha256_of("evidence.txt")
custody_log = []
custody_log.append(f"{datetime.datetime.now()} - SEIZED by Officer A - hash={seizure_hash}")

# Step 2: Evidence is later handed to a forensic analyst; verify integrity
handoff_hash = sha256_of("evidence.txt")
if handoff_hash == seizure_hash:
    custody_log.append(f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity VERIFIED")
else:
    custody_log.append(f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity FAILED (tampered!)")

print("=== Chain of Custody Log ===")
for entry in custody_log:
    print(entry)

# Experiment 3


In [ ]:
import re
def check_email(sender, subject, body):
flags = []
if re.search(r"(urgent|verify your account|suspended|click here)", body,
re.IGNORECASE):
flags.append("Urgency/pressure language detected")
if re.search(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", body):
flags.append("Raw IP address link found in body")
domain = sender.split("@")[-1]
if any(k in domain.lower() for k in ["secure","verify","update"]) and "@gmail" not
in sender:
flags.append("Suspicious sender domain naming pattern")
return flags
emails = [
("support@paypal.com", "Your monthly statement", "Please find your statement
attached."),
("alert@paypal-secure-verify.com", "URGENT: Verify your account", "Click here:
http://192.168.10.5/login"),
]
for sender, subject, body in emails:
issues = check_email(sender, subject, body)
verdict = "PHISHING SUSPECTED" if issues else "Looks legitimate"
print(f"\nFrom: {sender}\nSubject: {subject}\nVerdict: {verdict}")
for i in issues:
print(" -", i)

# Experiment 4


In [ ]:
import socket
domains = ["www.google.com", "www.python.org", "notarealdomain12345.com"]
print("OSINT Domain Reconnaissance")
for d in domains:
    try:
        ip = socket.gethostbyname(d)
        print(f"{d:30s} -> {ip}")
    except socket.gaierror:
        print(f"{d:30s} -> Could not resolve (invalid/unreachable)")

# Experiment 5


In [ ]:
import datetime
def investigation_report(case_id, evidence_list):
    stages = {
        "Identification": f"Incident reported for case {case_id}. Devices/logs identified for review.",
        "Collection": f"{len(evidence_list)} item(s) collected: {', '.join(evidence_list)}",
        "Preservation": "All items hashed (SHA-256) and stored in a write-protected evidence folder.",
        "Analysis": "Log files and file metadata examined for indicators of compromise.",
        "Reporting": "Findings compiled into a structured forensic report for legal review.",
    }
    print(f"=== Investigation Lifecycle: Case {case_id} ===")
    print(f"Generated: {datetime.datetime.now()}\n")
    for stage, detail in stages.items():
        print(f"[{stage}]")
        print(f" {detail}\n")
investigation_report("CASE-2026-014", ["laptop_disk_image.dd", "router_traffic.pcap", "email_headers.txt"])

# Experiment 6


In [ ]:
import hashlib
def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()
# Create a sample "disk" file (simulating a small storage device)
with open("original_disk.img", "wb") as f:
    f.write(b"HEADER" + bytes(range(256)) * 4 + b"FOOTER")
# Step 1: Create a bit-for-bit forensic copy (bit-stream image)
with open("original_disk.img", "rb") as src, open("forensic_copy.img", "wb") as dst:
    dst.write(src.read())
# Step 2: Verify the copy is identical using hashing
original_hash = sha256_of("original_disk.img")
copy_hash = sha256_of("forensic_copy.img")
print("Original image hash:", original_hash)
print("Forensic copy hash :", copy_hash)
print("Match:", "VERIFIED - exact bit-stream copy" if original_hash == copy_hash else "MISMATCH - copy corrupted")

# Experiment 7


In [ ]:
SIGNATURES = {
    b"\xFF\xD8\xFF": "JPEG image",
    b"\x89PNG": "PNG image",
    b"%PDF": "PDF document",
    b"PK\x03\x04": "ZIP archive (or .docx/.xlsx)",
}

def identify_file(path):
    with open(path, "rb") as f:
        header = f.read(8)
        for sig, filetype in SIGNATURES.items():
            if header.startswith(sig):
                return filetype
    return "Unknown file type"

# Create sample files with fake/renamed extensions to test signature detection
with open("photo.txt", "wb") as f: # renamed JPEG
    f.write(b"\xFF\xD8\xFF\xE0" + b"\x00" * 20)

with open("document.dat", "wb") as f: # renamed PDF
    f.write(b"%PDF-1.4" + b"\x00" * 20)

for filename in ["photo.txt", "document.dat"]:
    print(f"{filename:15s} -> Actual type: {identify_file(filename)}")

# Experiment 8


In [ ]:
import os
# Step 1: Create a file, then "delete" it (simulating accidental/malicious deletion)
with open("secret_note.txt", "w") as f:
    f.write("MEETING_POINT: warehouse 7, 11pm")
with open("secret_note.txt", "rb") as f:
    original_bytes = f.read()
os.remove("secret_note.txt")
print("File 'secret_note.txt' deleted.")
print("Exists on disk now?", os.path.exists("secret_note.txt"))
# Step 2: Simulate raw disk scanning - in real forensics, tools like Autopsy
# scan unallocated space for byte patterns. Here we search a "disk buffer"
# (which still holds the bytes) for the recoverable content.
disk_buffer = original_bytes + b"\x00" * 50 # unallocated space padding
marker = b"MEETING_POINT"
index = disk_buffer.find(marker)
if index != -1:
    recovered = disk_buffer[index:].split(b"\x00")[0]
    with open("recovered_note.txt", "wb") as f:
        f.write(recovered)
    print("Recovered content:", recovered.decode())
else:
    print("No recoverable data found.")

# Experiment 9


In [ ]:
import re
from collections import Counter

# Create the sample system log so the script runs standalone
with open("login_attempts.log", "w") as f:
    f.write("2026-07-08 09:00:01 LOGIN SUCCESS user=alice ip=10.0.0.5\n")
    f.write("2026-07-08 09:01:15 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:20 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:25 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:30 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:01:35 LOGIN FAILED user=admin ip=203.0.113.99\n")
    f.write("2026-07-08 09:02:00 LOGIN SUCCESS user=bob ip=10.0.0.8\n")

failed_by_ip = Counter()
with open("login_attempts.log") as f:
    for line in f:
        if "LOGIN FAILED" in line:
            match = re.search(r"ip=(\S+)", line)
            if match:
                failed_by_ip[match.group(1)] += 1

print("System log forensic analysis - failed logins by source IP:")
for ip, count in failed_by_ip.items():
    print(f" {ip}: {count} failed attempts")
    if count >= 5:
        print(f" -> Evidence of brute-force intrusion attempt from {ip}")

# Experiment 10


In [ ]:
raw_header = """Delivered-To: victim@example.com
Received: from mail-relay-99.suspicious-host.ru (unknown [45.33.32.156])
by mx.example.com; Tue, 08 Jul 2026 09:15:00 +0000
From: "Bank Support" <support@paypal-secure-verify.com>
Reply-To: attacker@totallynotscam.net
Subject: Urgent: Verify your account
"""
def analyze_header(header_text):
    findings = []
    for line in header_text.splitlines():
        if line.startswith("Received:") and "suspicious" in line.lower():
            findings.append(f"Suspicious relay server found: {line.strip()}")
        if line.startswith("Reply-To:"):
            findings.append(f"Reply-To differs from sender - possible spoofing:\n{line.strip()}")
        if line.startswith("From:") and "-secure-verify" in line:
            findings.append(f"Sender domain uses suspicious naming: {line.strip()}")
    return findings

print("Email Header Forensic Analysis")
for finding in analyze_header(raw_header):
    print("-", finding)

# Experiment 11


In [ ]:
# Create a sample simulated packet log so this runs standalone
with open("traffic.log", "w") as f:
    f.write("10.0.0.9 -> 10.0.0.5:22 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:23 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:25 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:80 SYN\n")
    f.write("10.0.0.9 -> 10.0.0.5:443 SYN\n")
    f.write("10.0.0.20 -> 10.0.0.5:80 SYN\n")
    f.write("10.0.0.20 -> 10.0.0.5:80 ACK\n")

# Count distinct destination ports probed by each source IP
ports_by_source = {}
with open("traffic.log") as f:
    for line in f:
        parts = line.split()
        src = parts[0]
        dst_port = parts[2].split(":")[1]
        ports_by_source.setdefault(src, set()).add(dst_port)

print("Network traffic forensic analysis:")
for src, ports in ports_by_source.items():
    print(f" {src}: contacted {len(ports)} distinct port(s) -> {sorted(ports)}")
    if len(ports) >= 4:
        print(f" -> ALERT: {src} shows port-scanning behaviour")

# Experiment 12


In [ ]:
from PIL import Image
from PIL.ExifTags import TAGS
# Install once: pip install Pillow
def create_sample_image(path):
    # Creates a small test image (a real photo would already have EXIF data)
    img = Image.new("RGB", (100, 100), color="blue")
    img.save(path)
def show_metadata(path):
    img = Image.open(path)
    print("File:", path)
    print("Format:", img.format)
    print("Size:", img.size)
    print("Mode:", img.mode)
    exif_data = img._getexif() if hasattr(img, "_getexif") else None
    if exif_data:
        for tag_id, value in exif_data.items():
            tag = TAGS.get(tag_id, tag_id)
            print(f" {tag}: {value}")
    else:
        print(" No EXIF metadata found (this sample image has none).")
        print(" Real photos from phones/cameras typically contain GPS,")
        print(" device model, and timestamp data - valuable forensic evidence.")
create_sample_image("sample.jpg")
show_metadata("sample.jpg")

# Experiment 13


In [ ]:
import datetime
def generate_forensic_report(case_id, findings, investigator):
    lines = []
    lines.append("DIGITAL FORENSIC INVESTIGATION REPORT")
    lines.append(f"Case ID: {case_id}")
    lines.append(f"Investigator: {investigator}")
    lines.append(f"Report generated: {datetime.datetime.now()}")
    lines.append("-" * 45)
    lines.append("FINDINGS:")
    for i, finding in enumerate(findings, 1):
        lines.append(f" {i}. {finding}")
    lines.append("-" * 45)
    lines.append("CONCLUSION:")
    lines.append(" Evidence indicates unauthorized access consistent with a")
    lines.append(" brute-force attack. Recommend IP block and password reset.")
    lines.append(" This report is suitable for legal review and expert testimony.")
    return "\n".join(lines)

findings = [
    "203.0.113.99 attempted 5 failed logins within 20 seconds (log file evidence).",
    "SHA-256 hash of evidence file verified unchanged throughout custody.",
    "Email header traced to a spoofed domain via suspicious relay server.",
]
report = generate_forensic_report("CASE-2026-014", findings, "Analyst B")
print(report)
with open("forensic_report.txt", "w") as f:
    f.write(report)

# Experiment 14


# Experiment 15


In [ ]:
import hashlib
import os
import requests
from google.colab import userdata
import time

# --- File Hash Generation ---

def generate_file_hashes(filepath):
    """Generates MD5, SHA-1, and SHA-256 hashes for a given file."""
    hashes = {
        'md5': hashlib.md5(),
        'sha1': hashlib.sha1(),
        'sha256': hashlib.sha256()
    }

    with open(filepath, 'rb') as f:
        while chunk := f.read(4096):
            for h in hashes.values():
                h.update(chunk)

    return {name: h.hexdigest() for name, h in hashes.items()}

# Create a dummy file for demonstration
sample_filename = 'sample_malware_file.txt'
with open(sample_filename, 'w') as f:
    f.write('This is a simulated malware file content. Do not run this on a real system.')
    f.write('Adding more content to make the file larger and more realistic for hashing.')

print(f"Created sample file: {sample_filename}")

# Generate and display hashes for the sample file
file_hashes = generate_file_hashes(sample_filename)
print("\nHashes for {}:".format(sample_filename))
for hash_type, hash_value in file_hashes.items():
    print(f"  {hash_type.upper()}: {hash_value}")

# os.remove(sample_filename) # Uncomment to clean up the sample file

# --- VirusTotal API Query ---

# Install the requests library if not already installed
!pip install requests --quiet

# Get the VirusTotal API key - User requested to hardcode it.
# IMPORTANT: Replace "YOUR_VIRUSTOTAL_API_KEY_HERE" with your actual VirusTotal API key.
# Hardcoding API keys is generally not recommended for security reasons.
VIRUSTOTAL_API_KEY = "YOUR_VIRUSTOTAL_API_KEY_HERE"

def query_virustotal(hash_value, api_key):
    """Queries the VirusTotal API for a given file hash."""
    url = f"https://www.virustotal.com/api/v3/files/{hash_value}"
    headers = {
        "x-apikeys": api_key,
        "Accept": "application/json"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
        return response.json()
    except requests.exceptions.HTTPError as http_err:
        if response.status_code == 404:
            print(f"  Hash not found on VirusTotal (404 Not Found).")
            return None
        elif response.status_code == 429:
            print(f"  Rate limit exceeded (429 Too Many Requests). Please wait and try again.")
            return None
        else:
            print(f"  HTTP error occurred: {http_err}")
            return None
    except Exception as err:
        print(f"  An error occurred: {err}")
        return None

print("\nQuerying VirusTotal for each generated hash...")

if not VIRUSTOTAL_API_KEY or VIRUSTOTAL_API_KEY == "71b4f79b55eeafbe1dac8ba2128a6ea059b3f7b11a933446d135f99f51bd815c":
    print("Error: VIRUSTOTAL_API_KEY is not set or is the placeholder. Please update it to proceed.")
else:
    for hash_type, hash_value in file_hashes.items():
        print(f"\nChecking {hash_type.upper()} hash: {hash_value}")
        vt_report = query_virustotal(hash_value, VIRUSTOTAL_API_KEY)

        if vt_report:
            data = vt_report.get('data')
            if data:
                attributes = data.get('attributes')
                if attributes:
                    last_analysis_stats = attributes.get('last_analysis_stats')
                    if last_analysis_stats:
                        harmless = last_analysis_stats.get('harmless', 0)
                        malicious = last_analysis_stats.get('malicious', 0)
                        undetected = last_analysis_stats.get('undetected', 0)

                        print(f"  VirusTotal Analysis Summary:")
                        print(f"    Malicious: {malicious}")
                        print(f"    Harmless: {harmless}")
                        print(f"    Undetected: {undetected}")
                        print(f"    Total engines: {harmless + malicious + undetected}")

                        if malicious > 0:
                            print(f"  * This hash is detected as MALICIOUS by {malicious} engines. *")
                        else:
                            print(f"  * This hash is not detected as malicious by any engine. *")
                    else:
                        print("  No last_analysis_stats found in report.")
                else:
                    print("  No attributes found in report data.")
            else:
                print("  No data found in VirusTotal report.")
        else:
            print(f"  Could not retrieve report for {hash_type.upper()} hash.")

        time.sleep(1) # Add a small delay to avoid hitting API rate limits


In [5]:
import os
import hashlib

# Experiment 16


In [12]:
import os
import hashlib

evidence_dir = "digital_evidence"
os.makedirs(evidence_dir, exist_ok=True)
sample_evidence = {
    "email_log.txt": """From: attacker@mail.com\nTo: victim@mail.com\nSubject: Invoice\nPlease pay immediately.""",
    "browser_history.txt": "http://malicious-site.com/login\nhttp://bank.com/transfer",
    "chat_message.txt": "Hey, did you send the file?",
    "system_log.txt": "2026-07-24 10:15:32 - USB device connected: E:\\.",
}

for filename, content in sample_evidence.items():
    with open(os.path.join(evidence_dir, filename), "w") as f:
        f.write(content)

def catalog_evidence(folder):
    catalog = []
    for fname in sorted(os.listdir(folder)):
        path = os.path.join(folder, fname)
        stat = os.stat(path)
        with open(path, "rb") as f:
            data = f.read()
        catalog.append({
            "filename": fname,
            "size_bytes": stat.st_size,
            "md5": hashlib.md5(data).hexdigest(),
        })
    return catalog

evidence_catalog = catalog_evidence(evidence_dir)
for item in evidence_catalog:
    print(item)

{'filename': 'browser_history.txt', 'size_bytes': 56, 'md5': '7804eb386588e70e52f8e44985aae9ab'}
{'filename': 'chat_message.txt', 'size_bytes': 27, 'md5': 'e5f0758dbf1fe3c48005ea2560fa5279'}
{'filename': 'email_log.txt', 'size_bytes': 84, 'md5': '94d472f8d614b91afc7ab2f56ddb435d'}
{'filename': 'system_log.txt', 'size_bytes': 48, 'md5': 'c43bdc9f6e214a2e73df493bbe761ae7'}


# Experiment 17


In [6]:
!pip install psutil --quiet

In [7]:
import psutil
import os

In [10]:
import psutil
import os

def capture_volatile_evidence():
    processes = [p.info for p in psutil.process_iter(['pid', 'name'])]
    return {
        "running_process_count": len(processes),
        "sample_processes": processes[:5],
    }
def capture_nonvolatile_evidence(folder):
    files = sorted(os.listdir(folder))
    return {"disk_files": files, "file_count": len(files)}
volatile_snapshot = capture_volatile_evidence()
nonvolatile_snapshot = capture_nonvolatile_evidence(evidence_dir)
print("Volatile evidence -> running processes:",
volatile_snapshot["running_process_count"])
print("Non-volatile evidence -> disk files:", nonvolatile_snapshot)

FileNotFoundError: [Errno 2] No such file or directory: 'digital_evidence'

# Experiment 18


In [ ]:
os.makedirs("acquisition_demo/source", exist_ok=True)
source = "acquisition_demo/source"
with open(os.path.join(source, "report.docx"), "w") as f:
    f.write("Confidential quarterly report.")
with open(os.path.join(source, "photo.jpg"), "w") as f:
    f.write("FAKEJPEGDATA")
raw_disk = b"REPORTDOCX_CONTENT" + b"\x00" * 20 + b"DELETED_INVOICE_DATA" + b"\x00" * 10
def physical_acquisition(raw_bytes):
    return bytes(raw_bytes) # every bit: used + unused + deleted
def logical_acquisition(folder):
    return {f: open(os.path.join(folder, f), "rb").read() for f in os.listdir(folder)}
def sparse_acquisition(folder, targets):
    return {f: open(os.path.join(folder, f), "rb").read() for f in targets if f in os.listdir(folder)}

physical_image = physical_acquisition(raw_disk)
logical_image = logical_acquisition(source)
sparse_image = sparse_acquisition(source, ["report.docx"])
print("Physical image size:", len(physical_image), "bytes")
print("Logical image files:", list(logical_image.keys()))
print("Sparse image files:", list(sparse_image.keys()))

In [8]:
import time
import psutil
import json

# Experiment 19


In [11]:
def live_acquisition():
    return {
        "timestamp": time.time(),
        "running_processes": len(psutil.pids()),
        "cpu_percent": psutil.cpu_percent(interval=0.1),
    }
def dead_acquisition(snapshot_file):
    with open(snapshot_file) as f:
        return json.load(f)
dead_snapshot_path = "system_snapshot.json"
static_snapshot = {"hard_disk_files": ["a.txt", "b.txt"], "ram_data": None}
with open(dead_snapshot_path, "w") as f:
    json.dump(static_snapshot, f)
live_result = live_acquisition()
dead_result = dead_acquisition(dead_snapshot_path)
print("Live acquisition:", live_result)
print("Dead acquisition:", dead_result)

Live acquisition: {'timestamp': 1785390015.668682, 'running_processes': 15, 'cpu_percent': 55.0}
Dead acquisition: {'hard_disk_files': ['a.txt', 'b.txt'], 'ram_data': None}


# Experiment 20


In [1]:
raw_disk_full = b"FILE1DATA" + b"\x00" * 15 + b"DELETED_FILE_DATA" + b"\x00" * 15 + b"FREE_SPACE_00000"
def create_forensic_image(raw_bytes):
    return bytes(raw_bytes) # bit-for-bit: files + deleted data + free space
def create_duplication(raw_bytes, active_regions):
    return b"".join(raw_bytes[start:end] for start, end in active_regions)
active_regions = [(0, 9)] # only the live "FILE1DATA" region
forensic_image = create_forensic_image(raw_disk_full)
duplication_copy = create_duplication(raw_disk_full, active_regions)
print("Forensic image size:", len(forensic_image))
print("Duplication size:", len(duplication_copy))

Forensic image size: 72
Duplication size: 9


# Experiment 21


In [14]:
import os
import hashlib

original_path = "original_evidence.bin"
with open(original_path, "wb") as f:
    f.write(os.urandom(1024)) # simulate a small storage device

def bit_stream_copy(src_path, dst_path):
    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        dst.write(src.read())

copy_path = "bitstream_copy.bin"
bit_stream_copy(original_path, copy_path)

def sha256_of_file(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

original_hash = sha256_of_file(original_path)
copy_hash = sha256_of_file(copy_path)
print("Original hash:", original_hash)
print("Copy hash: ", copy_hash)

Original hash: cca0f6cb894a9a66dc6f3b7969f2dcf65ec24faff5da03e600e692425c94dafb
Copy hash:  cca0f6cb894a9a66dc6f3b7969f2dcf65ec24faff5da03e600e692425c94dafb


# Experiment 22


In [16]:
import hashlib

def compute_hashes(data: bytes):
    return {
        "MD5": hashlib.md5(data).hexdigest(),
        "SHA1": hashlib.sha1(data).hexdigest(),
        "SHA256": hashlib.sha256(data).hexdigest(),
    }

original_data = b"hello"
tampered_data = b"Hello"
hashes_original = compute_hashes(original_data)
hashes_tampered = compute_hashes(tampered_data)
hashes_original_repeat = compute_hashes(original_data)
print("hello ->", hashes_original)
print("Hello ->", hashes_tampered)

hello -> {'MD5': '5d41402abc4b2a76b9719d911017c592', 'SHA1': 'aaf4c61ddcc5e8a2dabede0f3b482cd9aea9434d', 'SHA256': '2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824'}
Hello -> {'MD5': '8b1a9953c4611296a827abf8c47804d7', 'SHA1': 'f7ff9e8b7bb2e09b70935a5d785e0cc5d9d0abf0', 'SHA256': '185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969'}


# Experiment 23


In [21]:
import os

def make_fake_jpeg(payload: bytes):
    return b"\xff\xd8\xff" + payload + b"\xff\xd9"

jpeg1 = make_fake_jpeg(b"PHOTO_OF_SUSPECT_CAR")
jpeg2 = make_fake_jpeg(b"CCTV_FRAME_CAPTURE")
raw_disk_blob = os.urandom(30) + jpeg1 + os.urandom(40) + jpeg2 + os.urandom(20)

def carve_jpegs(blob: bytes):
    recovered = []
    start_marker, end_marker = b"\xff\xd8\xff", b"\xff\xd9"
    pos = 0
    while True:
        start = blob.find(start_marker, pos)
        if start == -1:
            break
        end = blob.find(end_marker, start)
        if end == -1:
            break
        end += len(end_marker)
        recovered.append(blob[start:end])
        pos = end
    return recovered

recovered_files = carve_jpegs(raw_disk_blob)

# Experiment 24


In [22]:
class FATFileEntry:
    def __init__(self, name, size):
        self.name = name
        self.size = size
        # FAT intentionally has no permissions or journal attribute
class NTFSFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"

class EXTFileEntry:
    def __init__(self, name, size, permissions="rw-r--r--"):
        self.name = name
        self.size = size
        self.permissions = permissions
        self.journal_entry = f"WRITE {name} {size} bytes"
fat_file = FATFileEntry("data.txt", 1024)
ntfs_file = NTFSFileEntry("data.txt", 1024)
ext_file = EXTFileEntry("data.txt", 1024)
print("FAT has permissions attribute:", hasattr(fat_file, "permissions"))
print("NTFS has permissions attribute:", hasattr(ntfs_file, "permissions"))
print("EXT has permissions attribute:", hasattr(ext_file, "permissions"))

FAT has permissions attribute: False
NTFS has permissions attribute: True
EXT has permissions attribute: True


# Experiment 25


In [24]:
CLUSTER_SIZE = 4096 # typical disk cluster size in bytes
def calculate_slack_space(file_size, cluster_size=CLUSTER_SIZE):
    clusters_needed = -(-file_size // cluster_size) # ceiling division
    allocated_space = clusters_needed * cluster_size
    slack_space = allocated_space - file_size
    return allocated_space, slack_space
file_size = 5000 # bytes
allocated, slack = calculate_slack_space(file_size)
print(f"File size: {file_size} bytes -> allocated: {allocated} bytes, slack space:\n{slack} bytes")

File size: 5000 bytes -> allocated: 8192 bytes, slack space:
3192 bytes
